<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/project-2-llm-evaluation-suite/06b_regression_alarm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 6b: Weekly Regression Alarm and Production-Sampled Trace Dataset

**Goal:** Build a regression alarm that compares current evaluation scores against a prior baseline and flags meaningful drops, plus a production-sampling mechanism for selecting traces for human annotation. This is the last piece before Phase 7's synthesis.

**Design decision:** the alarm logic itself (threshold comparison, drift detection) and the sampling logic are both deterministic. Neither needs a model call, so both are built as real, working code from the start, no `SIMULATED_OUTPUT` branch needed for the core logic itself, the same approach used in Phase 5c's ATLAS mapping.

**Honest limitation, stated upfront:** a regression alarm's entire purpose is comparing *this week* against *prior weeks*. Project 2 has only ever run once, there is no real second data point yet. This notebook uses Phase 6a's saved scores as the one real baseline that exists, and constructs one clearly-labeled demonstration comparison point to prove the alarm logic actually fires when it should and stays quiet when it shouldn't. That demonstration point is not invented history presented as real, it is stated plainly as a synthetic test case for the alarm's own logic.

**Tools:** Langfuse v4, pandas

**Date:** July 2026

**Status:** In progress. Alarm and sampling logic are real. Real week-over-week comparison requires this suite to actually run more than once over time, which has not happened yet.

In [1]:
# Cell 2: Mount Drive and load Phase 6a as the real baseline

from google.colab import drive
drive.mount('/content/drive')

import os, json

DRIVE_PATH = "/content/drive/MyDrive/python-ai-governance-p2/data/"

phase6a_path = DRIVE_PATH + "phase06a_langfuse_custom_scores_results.json"
if os.path.exists(phase6a_path):
    with open(phase6a_path) as f:
        phase6a = json.load(f)
    print("Phase 6a results confirmed.")
    print(f"  Sources wired: {phase6a['trace_count']}")
    print(f"  Real: 0 | Simulated/Mixed: {phase6a['trace_count']} "
          f"(per Phase 6a's own findings)")
else:
    print("WARNING: Phase 6a results not found.")
    print(f"Expected: {phase6a_path}")
    print("Run 06a_langfuse_custom_scores.ipynb first.")

Mounted at /content/drive
Phase 6a results confirmed.
  Sources wired: 9
  Real: 0 | Simulated/Mixed: 9 (per Phase 6a's own findings)


In [2]:
# Cell 3: Install packages
# Same as Phase 5c: the alarm and sampling logic are deterministic and need
# no LLM API at all. Langfuse kept for consistency with every phase's trace
# logging convention.

!pip install langfuse pandas --quiet

print("Packages installed.")
print("No Gemini or Claude client needed in this notebook's core logic.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 11.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.
Packages installed.
No Gemini or Claude client needed in this notebook's core logic.


In [3]:
# Cell 4: Simulated output flag and Langfuse client
# Same decoupling as Phase 6a: this flag only governs whether Langfuse
# traces are sent live. The alarm logic and sampling logic below are real,
# deterministic code regardless of this flag's value.

SIMULATED_OUTPUT = True

from google.colab import userdata

if not SIMULATED_OUTPUT:
    from langfuse import Langfuse
    langfuse = Langfuse(
        public_key=userdata.get('LANGFUSE_PUBLIC_KEY'),
        secret_key=userdata.get('LANGFUSE_SECRET_KEY'),
        host="https://cloud.langfuse.com"
    )
    print("Langfuse client initialised.")
else:
    print("[SIMULATED] Langfuse client not initialised (trace logging only).")
    print(f"SIMULATED_OUTPUT = {SIMULATED_OUTPUT}")

print()
print("Reminder: the regression alarm and sampling logic below are real,")
print("deterministic code. This flag affects only whether alarm-trigger")
print("events get logged to a live Langfuse dashboard.")

[SIMULATED] Langfuse client not initialised (trace logging only).
SIMULATED_OUTPUT = True

Reminder: the regression alarm and sampling logic below are real,
deterministic code. This flag affects only whether alarm-trigger
events get logged to a live Langfuse dashboard.


In [4]:
# Cell 5: Regression alarm logic
# Real, deterministic comparison logic, no model call involved. Compares a
# "current" set of scores against a "baseline" set and flags any metric
# that dropped more than the configured threshold.

REGRESSION_THRESHOLD = 0.05  # a 5-percentage-point drop triggers an alarm

def check_regression(current_scores: dict, baseline_scores: dict,
                      threshold: float = REGRESSION_THRESHOLD) -> list:
    """Compares current vs baseline headline scores per phase.
    Returns a list of alarm events for any metric that dropped by more
    than `threshold`. This function is real and permanent, not a
    placeholder, it will run identically on real data once real data
    exists."""
    alarms = []
    for phase_key in baseline_scores:
        if phase_key not in current_scores:
            continue
        baseline_val = baseline_scores[phase_key]["headline_score"]
        current_val = current_scores[phase_key]["headline_score"]
        drop = baseline_val - current_val
        if drop > threshold:
            alarms.append({
                "phase": phase_key,
                "metric": baseline_scores[phase_key]["headline_label"],
                "baseline_score": baseline_val,
                "current_score": current_val,
                "drop": round(drop, 4),
                "severity": "ALARM" if drop > (threshold * 2) else "WARNING",
            })
    return alarms


# The one real baseline that exists: Phase 6a's normalized scores.
baseline_scores = phase6a["normalized_scores"]

print(f"Regression alarm logic loaded. Threshold: {REGRESSION_THRESHOLD} "
      f"({REGRESSION_THRESHOLD:.0%} drop triggers a warning, "
      f"{REGRESSION_THRESHOLD*2:.0%} triggers an alarm).")
print(f"Baseline loaded from Phase 6a: {len(baseline_scores)} phases.")

Regression alarm logic loaded. Threshold: 0.05 (5% drop triggers a warning, 10% triggers an alarm).
Baseline loaded from Phase 6a: 9 phases.


In [12]:
# Cell 7: Production-sampled trace dataset, with a synthetic case-level
# regression demonstration.
#
# Two distinct things happen in this cell, kept clearly separate:
# 1. REAL: sampling actual cases from actual saved results (self-contained,
#    loads its own source files directly, per the earlier fix).
# 2. SYNTHETIC DEMONSTRATION: since no sampled case has ever been re-run
#    (each ran exactly once), there is no real "before" for any individual
#    case, only Cell 6's phase-level demonstration exists so far. This
#    section extends that same honest pattern to case level: each sampled
#    case's REAL current score is compared against a SYNTHETIC baseline
#    (a clean-pass reference point, not a real prior measurement of that
#    case), to prove the same alarm mechanism would also work at case
#    granularity. This is stated plainly, not folded quietly into the
#    output as if it were real history.

import os, json, random

random.seed(42)

SAMPLE_SOURCES = {
    "phase03a": "phase03a_deepeval_rag_results.json",
    "phase04a": "phase04a_aspect_critic_results.json",
    "phase05a": "phase05a_promptfoo_owasp_results.json",
    "phase05b": "phase05b_promptfoo_owasp_agentic_results.json",
}

loaded_sources = {}
for key, filename in SAMPLE_SOURCES.items():
    path = DRIVE_PATH + filename
    if os.path.exists(path):
        with open(path) as f:
            loaded_sources[key] = json.load(f)
    else:
        loaded_sources[key] = None
        print(f"WARNING: {key} not found at {path}, excluded.")


def build_sample_pool() -> list:
    pool = []
    if loaded_sources.get("phase03a"):
        for case in loaded_sources["phase03a"]["per_case_results"]:
            pool.append({"source": "phase03a", "case_id": case["id"],
                         "outcome": case["actual_outcome"],
                         "simulated": loaded_sources["phase03a"]["simulated"]})
    if loaded_sources.get("phase04a"):
        for case in loaded_sources["phase04a"]["per_sample_results"]:
            pool.append({"source": "phase04a", "case_id": case["id"],
                         "outcome": case["actual_routing"],
                         "simulated": loaded_sources["phase04a"]["simulated"]})
    if loaded_sources.get("phase05a"):
        for case in loaded_sources["phase05a"]["per_case_results"]:
            pool.append({"source": "phase05a", "case_id": case["id"],
                         "outcome": "MISSED" if not case["detected"] else "DETECTED",
                         "simulated": loaded_sources["phase05a"]["simulated"]})
    if loaded_sources.get("phase05b"):
        for case in loaded_sources["phase05b"]["per_case_results"]:
            pool.append({"source": "phase05b", "case_id": case["id"],
                         "outcome": "MISSED" if not case["detected"] else "DETECTED",
                         "simulated": loaded_sources["phase05b"]["simulated"]})
    return pool


def stratified_sample(pool: list, sample_size: int = 5) -> list:
    priority = [c for c in pool if c["outcome"] in ("BORDERLINE", "FAIL", "MISSED")]
    routine = [c for c in pool if c["outcome"] in ("PASS", "DETECTED")]
    by_source = {}
    for c in priority:
        by_source.setdefault(c["source"], []).append(c)
    sample = []
    sources_cycle = list(by_source.keys())
    while len(sample) < sample_size and any(by_source.values()):
        for src in sources_cycle:
            if by_source[src]:
                sample.append(by_source[src].pop(0))
                if len(sample) >= sample_size:
                    break
    remaining = sample_size - len(sample)
    if remaining > 0 and routine:
        sample += random.sample(routine, min(remaining, len(routine)))
    return sample


sample_pool = build_sample_pool()
production_sample = stratified_sample(sample_pool, sample_size=5)

print(f"REAL: sample pool built: {len(sample_pool)} cases across "
      f"03a, 04a, 05a, 05b.")
print(f"REAL: stratified sample selected: {len(production_sample)} cases.")
print()

# --- Real current score extraction, per source's actual data shape ---

def stratified_sample(pool: list, sample_size: int = 6,
                       min_routine: int = 1) -> list:
    """Same priority-first, source-balanced logic as before, but now
    reserves at least `min_routine` slots for routine (PASS/DETECTED)
    cases, so the demonstration can show the full ALARM/WARNING/OK range
    rather than only ever showing worst-case cases."""
    priority = [c for c in pool if c["outcome"] in ("BORDERLINE", "FAIL", "MISSED")]
    routine = [c for c in pool if c["outcome"] in ("PASS", "DETECTED")]

    priority_slots = sample_size - min_routine

    by_source = {}
    for c in priority:
        by_source.setdefault(c["source"], []).append(c)
    sample = []
    sources_cycle = list(by_source.keys())
    while len(sample) < priority_slots and any(by_source.values()):
        for src in sources_cycle:
            if by_source[src]:
                sample.append(by_source[src].pop(0))
                if len(sample) >= priority_slots:
                    break

    routine_slots = sample_size - len(sample)
    if routine_slots > 0 and routine:
        sample += random.sample(routine, min(routine_slots, len(routine)))
    return sample


production_sample = stratified_sample(sample_pool, sample_size=6, min_routine=1)

print(f"REAL: stratified sample selected (source-balanced, "
      f"with guaranteed routine case): {len(production_sample)} cases.")
print()

for c in production_sample:
    real_score, justification = get_real_current_score(c)
    drop = SYNTHETIC_CASE_BASELINE - real_score
    if drop > REGRESSION_THRESHOLD * 2:
        severity = "ALARM"
    elif drop > REGRESSION_THRESHOLD:
        severity = "WARNING"
    else:
        severity = "OK"

    print(f"[{severity}] {c['source']} / {c['case_id']}: {c['outcome']}")
    print(f"  synthetic_baseline={SYNTHETIC_CASE_BASELINE:.2f} -> "
          f"real_current={real_score:.2f} (drop={drop:.2f})")
    print(f"  Justification: {justification}")
    print()

# Purely synthetic addition: no real case in this dataset naturally falls
# in the WARNING band (drop between REGRESSION_THRESHOLD and 2x it) against
# SYNTHETIC_CASE_BASELINE. Rather than force a real case to look like
# something it isn't, one fabricated case is added here, explicitly
# labeled, matching Cell 6's own precedent for proving a severity tier
# the real data doesn't happen to exercise.

SYNTHETIC_WARNING_CASE = {
    "case_id": "SYNTHETIC_demo_only",
    "source": "none (fabricated for demonstration)",
    "outcome": "N/A, not a real evaluation outcome",
    "real_current": 0.91,
}

drop = SYNTHETIC_CASE_BASELINE - SYNTHETIC_WARNING_CASE["real_current"]
severity = "ALARM" if drop > REGRESSION_THRESHOLD * 2 else (
           "WARNING" if drop > REGRESSION_THRESHOLD else "OK")

print(f"[{severity}] {SYNTHETIC_WARNING_CASE['source']} / "
      f"{SYNTHETIC_WARNING_CASE['case_id']}: {SYNTHETIC_WARNING_CASE['outcome']}")
print(f"  synthetic_baseline={SYNTHETIC_CASE_BASELINE:.2f} -> "
      f"synthetic_current={SYNTHETIC_WARNING_CASE['real_current']:.2f} (drop={drop:.2f})")
print(f"  Justification: fully fabricated case, added only to prove the "
      f"WARNING tier of this mechanism fires correctly, since no real case "
      f"in the current 33-case dataset naturally falls in this band.")

REAL: sample pool built: 33 cases across 03a, 04a, 05a, 05b.
REAL: stratified sample selected: 5 cases.

REAL: stratified sample selected (source-balanced, with guaranteed routine case): 6 cases.

[ALARM] phase03a / tc_003: BORDERLINE
  synthetic_baseline=1.00 -> real_current=0.62 (drop=0.38)
  Justification: worst metric=contextual_recall: Expected output partially covered. Accountability detail missing from context.

[ALARM] phase04a / as_004: FAIL
  synthetic_baseline=1.00 -> real_current=0.40 (drop=0.60)
  Justification: aspect pass fraction, critical failures: ['harm_potential', 'correctness'], Wrong figures could cause deployer non-compliance.

[ALARM] phase03a / tc_004: FAIL
  synthetic_baseline=1.00 -> real_current=0.09 (drop=0.91)
  Justification: worst metric=answer_correctness: Factually incorrect. EUR 50M/10pct vs 35M/7pct.

[ALARM] phase04a / as_005: FAIL
  synthetic_baseline=1.00 -> real_current=0.20 (drop=0.80)
  Justification: aspect pass fraction, critical failures: ['